In [ ]:
!pip install langchain langgraph langchain-openai langchain-core langchain-community langchain-experimental fpdf pdfplumber

  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import AIMessage
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from pydantic import BaseModel, Field
from fpdf import FPDF
import random
import pdfplumber
import os
import requests
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pwd

/content


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY loaded: True


In [ ]:
# LLM 정의
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o",
                temperature=0)

In [ ]:
@tool
def read_pdf(file_path: str):
    """
    PDF 파일에서 텍스트를 추출하는 도구입니다.
    표 형식 또는 일반 텍스트가 포함된 PDF를 읽고 문자열로 반환합니다.

    file_path 예시: './report.pdf'
    """
    try:
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip() if text.strip() else "❌ PDF에서 텍스트를 추출할 수 없습니다."
    except Exception as e:
        return f"❌ PDF 읽기 오류: {str(e)}"

In [ ]:
@ tool
def write_pdf(content: str, filename: str = "output.pdf", summary: bool =True):
    """
    텍스트를 PDF 파일로 저장하는 도구입니다.
    PDF형태의 문서로 만들어야할 때 이 도구를 사용하세요.
    """

    if summary:
        prompt = PromptTemplate.from_template("""
                당신은 보고서를 작성하는 어시스턴트입니다. 당신에겐 문서 모음이 제공되고 이를 잘 분석하여 보고서를 작성하여야 합니다.
                아래의 content는 문서 모음입니다. 문서의 제목, 본문을 잘 판단하고 정리하여 요약합니다.
                항상 구조화된 출력을 제공하세요.
                항상 마지막엔 인사이트도 첨부합니다.

                content : {content}
                """)

        chain = prompt | llm

        content = chain.invoke({"content":content}).content

    else:
        pass

    font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    font_path = "./fonts/NotoSansKR.ttf"

    try:
        os.mkdir("./fonts/")
        response = requests.get(font_url)
        with open(font_path, "wb") as f:
            f.write(response.content)
    except:
        pass

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    font_path = "/content/fonts/NotoSansKR.ttf"  # <-- 여기에 실제 폰트 파일이 있어야 함

    try:
        pdf.add_font("NotoSans", "", font_path, uni=True)
        pdf.set_font("NotoSans", size=12)
    except:
        raise ValueError("한글 폰트를 등록할 수 없습니다.")

    for line in content.split("\n"):
        pdf.multi_cell(0, 10, line)
    pdf.output(f"./{filename}")

    return f"{filename} 저장 완료"

In [ ]:
# PDF 쓰기 도구

from fpdf import FPDF
from datetime import datetime

# 1. 기본 FPDF 클래스 상속해서 헤더/푸터 커스터마이징
class PDF(FPDF):
    def __init__(self, font_path="/content/fonts/NotoSansKR.ttf"):
        super().__init__()
        self.title_text = ""
        self.font_path = font_path

    def header(self):
        """페이지 상단에 표시될 헤더"""
        if self.title_text:
            self.set_font('NotoSans', 'B', 16)
            self.cell(0, 10, self.title_text, border=0, ln=1, align='C')
            self.ln(5)

    def footer(self):
        """페이지 하단에 표시될 푸터"""
        self.set_y(-15)
        self.set_font('NotoSans', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

# 2. PDF 생성 함수
def create_structured_pdf(title, summary, content, filename):
    """
    제목, 요약, 내용을 포함한 구조화된 PDF 생성

    Args:
        title (str): 문서 제목
        summary (str): 문서 요약
        content (str): 본문 내용
        filename (str): 저장할 파일명 (예: "report.pdf")
    """

    # 한글 지원을 위해 폰트 설정 (기본 Arial은 한글 미지원)
    # 실제 사용시에는 한글 폰트 파일 경로 필요
    # pdf.add_font('NanumGothic', '', 'NanumGothic.ttf', uni=True)
    # pdf.set_font('NanumGothic', '', 12)

    # 임시로 Arial 사용 (한글은 깨질 수 있음)
    # pdf.set_font('Arial', '', 12)

    # 1. 폰트 폴더 및 파일 준비
    font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    font_path = "./fonts/NotoSansKR.ttf"

    try:
        os.mkdir("./fonts/")
        response = requests.get(font_url)
        with open(font_path, "wb") as f:
            f.write(response.content)
    except:
        pass

    # 2. PDF 생성 (커스텀 헤더/푸터 + 폰트 설정)
    # font_path = "/content/fonts/NotoSansKR.ttf"  # <-- 여기에 실제 폰트 파일이 있어야 함
    pdf = PDF(font_path=font_path)
    pdf.title_text = title

    try:
        # 한글 폰트 등록 및 기본 폰트 설정
        pdf.add_font("NotoSans", "", font_path, uni=True)
        pdf.set_font("NotoSans", size=12)
        pdf.add_font("NotoSans", "B", font_path,  uni=True)
        pdf.add_font("NotoSans", "I", font_path,  uni=True)
    except:
        raise ValueError("한글 폰트를 등록할 수 없습니다.")


    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # 제목 (이미 헤더에 표시되므로 여기서는 생략 가능)
    # pdf.set_font('Arial', 'B', 20)
    # pdf.cell(0, 10, title, ln=1, align='C')
    # pdf.ln(10)

    # 작성일시
    # pdf.set_font('Arial', 'I', 10)
    pdf.set_font("NotoSans", size=10)
    pdf.cell(0, 8, f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=1)
    pdf.ln(5)

    # 요약 섹션
    if summary:
        pdf.set_font('NotoSans', 'B', 14)
        pdf.cell(0, 10, "Summary", ln=1)
        pdf.set_font('NotoSans', '', 11)
        pdf.multi_cell(0, 6, summary)
        pdf.ln(5)

    # 구분선
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    # 본문 내용
    pdf.set_font('NotoSans', 'B', 14)
    pdf.cell(0, 10, "Content", ln=1)
    pdf.set_font('NotoSans', '', 11)
    pdf.multi_cell(0, 6, content)

    # PDF 파일 저장
    pdf.output(filename)
    print(f"✅ PDF saved: {filename}")
    return filename




In [ ]:
@ tool
def create_formated_pdf(title, summary, content, filename):
    """제목/요약/본문 내용을 받아 PDF 파일을 생성하고, 생성된 파일 경로를 반환하는 도구."""
    filename = create_structured_pdf(title, summary, content, filename)
    return filename

In [ ]:
# 툴 정의
# TavilySearchResults : 웹 검색 도구
# PythonAstREPLTool : 파이썬 코드 실행 도구
# write_pdf : pdf 생성 도구
# read_pdf : pdf 읽기 도구
# file_delete : 파일 삭제 도구
# list_directory : 파일 목록 읽기 도구

tools = [create_formated_pdf, PythonAstREPLTool(), write_pdf, read_pdf, *FileManagementToolkit(
                                                                            selected_tools=["file_delete","list_directory"]).get_tools()]
formated_write_tool, code_tool, write_tool, read_tool, delete_tool, listdir_tool= tools

In [ ]:
# PDF 제목/요약/내용 쓰기 도구 예시
if __name__ == "__main__":
    create_formated_pdf.invoke({
        "title": "내 이야기 제목",
        "summary": "이 이야기의 요약입니다.",
        "content": "본문 내용이 여기에 들어갑니다.\n\n여러 줄도 가능합니다.",
        "filename": "/content/drive/MyDrive/output.pdf"
    })

✅ PDF saved: /content/drive/MyDrive/output.pdf
